In [ ]:
# ============================================================
# BLOCK 1 — Setup
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import gzip, io, time, collections
import pandas as pd

file_path_auth = "/content/drive/MyDrive/auth.txt.gz"
file_path_redteam = "/content/drive/MyDrive/redteam.txt.gz"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================
# BLOCK 2 — Load Red Team ground truth
# ============================================================
redteam_columns = ['time', 'user', 'source_computer', 'destination_computer']
redteam_lines = []
with gzip.open(file_path_redteam, 'rt', encoding='utf-8') as f:
    for line in f:
        redteam_lines.append(line.strip())

redteam_df = pd.read_csv(io.StringIO('\n'.join(redteam_lines)), header=None, names=redteam_columns)
redteam_df['time'] = pd.to_numeric(redteam_df['time'])

redteam_tuples = set(zip(redteam_df['time'].astype(int), redteam_df['user'],
                          redteam_df['source_computer'], redteam_df['destination_computer']))
print(f"Loaded {len(redteam_df)} Red Team events, time range {redteam_df['time'].min()}-{redteam_df['time'].max()}")

Loaded 749 Red Team events, time range 150885-2557047


In [ ]:
# ============================================================
# BLOCK 3 — Build attack clusters, pick a spread-out test set
# ============================================================
rt_sorted = redteam_df.sort_values('time').reset_index(drop=True)
gap = rt_sorted['time'].diff().fillna(0)
rt_sorted['cluster'] = (gap > 3600).cumsum()

clusters = rt_sorted.groupby('cluster')['time'].agg(['min', 'max', 'count'])
print(f"Total clusters found: {len(clusters)}")

# 6 clusters spread across the full timeline — not just the first/easiest ones
chosen_idx = sorted(set(
    int(round(x)) for x in
    [0, len(clusters)*0.2, len(clusters)*0.4, len(clusters)*0.6, len(clusters)*0.8, len(clusters)-1]
))
chosen = clusters.iloc[chosen_idx]
print(chosen)

Total clusters found: 35
             min      max  count
cluster                         
0         150885   156658     10
7         483455   491747      7
14       1066394  1082244    148
21       1328198  1328481      4
28       2364275  2373636      8
34       2557047  2557047      1


In [ ]:
# ============================================================
# BLOCK 4 — Trigger rule: FANOUT (the one that worked — use this as the MVP1 trigger)
# ============================================================
"""
PAD = 300
BASELINE_DURATION = 45000
MIN_DESTINATIONS = 5
RATIO_THRESHOLD = 5
MIN_ABSOLUTE_INCREASE = 6

cluster_defs = []
for _, row in chosen.iterrows():
    w_start, w_end = row['min'] - PAD, row['max'] + PAD
    b_start, b_end = w_start - BASELINE_DURATION, w_start
    contaminated = ((redteam_df['time'] >= b_start) & (redteam_df['time'] <= b_end)).any()
    cluster_defs.append({'w_start': w_start, 'w_end': w_end, 'b_start': b_start, 'b_end': b_end,
                          'contaminated': contaminated})
clean_clusters = [c for c in cluster_defs if not c['contaminated']]
max_time = max(c['w_end'] for c in clean_clusters)

baseline_sets = [collections.defaultdict(set) for _ in clean_clusters]
window_sets = [collections.defaultdict(set) for _ in clean_clusters]
window_rows = [[] for _ in clean_clusters]

with gzip.open(file_path_auth, 'rt', encoding='utf-8') as f:
    for line in f:
        fields = line.strip().split(',')
        if len(fields) != 9: continue
        t, su, du, sc, dc, atype, ltype, orient, result = fields
        t = int(t)
        if t > max_time: break
        if ltype != 'Network': continue
        for i, c in enumerate(clean_clusters):
            if c['b_start'] <= t < c['b_end']:
                baseline_sets[i][sc].add(dc)
            elif c['w_start'] <= t <= c['w_end']:
                window_sets[i][sc].add(dc)
                window_rows[i].append((t, su, sc, dc))

def get_flagged_computers(cluster_i):
    """Returns the set of computers the fanout trigger flags for a given cluster."""
    all_computers = set(baseline_sets[cluster_i]) | set(window_sets[cluster_i])
    flagged = set()
    for comp in all_computers:
        b_fanout = len(baseline_sets[cluster_i].get(comp, set()))
        w_fanout = len(window_sets[cluster_i].get(comp, set()))
        ratio = (w_fanout + 1) / (b_fanout + 1)
        if w_fanout >= MIN_DESTINATIONS and (ratio >= RATIO_THRESHOLD or (w_fanout - b_fanout) >= MIN_ABSOLUTE_INCREASE):
            flagged.add(comp)
    return flagged

    """

In [ ]:
# ============================================================
# BLOCK 4-RELOAD — load saved state instead of streaming (replaces Block 4)
# ============================================================
import pickle

STATE_FILE = "/content/drive/MyDrive/cyberflight/stream_state.pkl"

with open(STATE_FILE, "rb") as f:          # "rb" = read bytes
    state = pickle.load(f)                 # load = the opposite of dump

clean_clusters = state["clean_clusters"]
baseline_sets  = state["baseline_sets"]
window_sets    = state["window_sets"]
window_rows    = state["window_rows"]

s = state["settings"]
PAD, BASELINE_DURATION = s["PAD"], s["BASELINE_DURATION"]
MIN_DESTINATIONS, RATIO_THRESHOLD, MIN_ABSOLUTE_INCREASE = (
    s["MIN_DESTINATIONS"], s["RATIO_THRESHOLD"], s["MIN_ABSOLUTE_INCREASE"])

print(f"Reloaded {len(clean_clusters)} clusters, "
      f"{sum(len(r) for r in window_rows):,} window rows")

Reloaded 4 clusters, 7,485,453 window rows


In [ ]:
# ============================================================
# BLOCK 4-RULE — fanout trigger (copied unchanged from Block 4)
# ============================================================
def get_flagged_computers(cluster_i):
    """Returns the set of computers the fanout trigger flags for a given cluster."""
    all_computers = set(baseline_sets[cluster_i]) | set(window_sets[cluster_i])
    flagged = set()
    for comp in all_computers:
        b_fanout = len(baseline_sets[cluster_i].get(comp, set()))
        w_fanout = len(window_sets[cluster_i].get(comp, set()))
        ratio = (w_fanout + 1) / (b_fanout + 1)
        if w_fanout >= MIN_DESTINATIONS and (ratio >= RATIO_THRESHOLD or (w_fanout - b_fanout) >= MIN_ABSOLUTE_INCREASE):
            flagged.add(comp)
    return flagged

In [ ]:
"""
# ============================================================
# BLOCK 4b
# ============================================================
import pickle, os

SAVE_DIR = "/content/drive/MyDrive/cyberflight"
STATE_FILE = f"{SAVE_DIR}/stream_state.pkl"
os.makedirs(SAVE_DIR, exist_ok=True)

state = {
    "clean_clusters": clean_clusters,
    "baseline_sets": [dict(d) for d in baseline_sets],
    "window_sets":   [dict(d) for d in window_sets],
    "window_rows":   window_rows,
    "settings": {"PAD": PAD, "BASELINE_DURATION": BASELINE_DURATION,
                 "MIN_DESTINATIONS": MIN_DESTINATIONS,
                 "RATIO_THRESHOLD": RATIO_THRESHOLD,
                 "MIN_ABSOLUTE_INCREASE": MIN_ABSOLUTE_INCREASE},
}

with open(STATE_FILE, "wb") as f:
    pickle.dump(state, f)

print(f"Saved to {STATE_FILE}: {os.path.getsize(STATE_FILE)/1e6:.1f} MB")
"""

_IncompleteInputError: incomplete input (2654010139.py, line 1)

In [ ]:
# ============================================================
# BLOCK 5 — Grade the trigger (sanity check before moving to MVP1)
# ============================================================

"""
this result is after the block-4"""
total_tp = total_fp = total_fn = total_tn = 0
for i, c in enumerate(clean_clusters):
    flagged = get_flagged_computers(i)
    for (t, su, sc, dc) in window_rows[i]:
        is_redteam = (t, su, sc, dc) in redteam_tuples
        is_flagged = sc in flagged
        if is_flagged and is_redteam: total_tp += 1
        elif is_flagged and not is_redteam: total_fp += 1
        elif not is_flagged and is_redteam: total_fn += 1
        else: total_tn += 1

precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0
recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0
print(f"Fanout trigger across {len(clean_clusters)} clean clusters")
print(f"TP={total_tp} FP={total_fp} FN={total_fn} TN={total_tn}")
print(f"Precision={precision:.4f} Recall={recall:.4f}")

Fanout trigger across 4 clean clusters
TP=164 FP=432063 FN=4 TN=7053222
Precision=0.0004 Recall=0.9762


In [ ]:
# ============================================================
# BLOCK 6 — Known limitation (document, don't fix in MVP1)
# ============================================================
# Cluster 21 (time 1328198-1328781): same-edge REPETITION burst — a known,
# already-seen (source_computer, destination_computer) pair reused rapidly.
# Neither fanout nor edge-novelty catches this pattern. Documented as a
# scoped-out Phase 2 signal, not fixed here.
print("Known gap: repetition-burst pattern (cluster 21) — not covered by current trigger.")

Step B: Cluster numbers

In [ ]:
# ============================================================
# BLOCK 4-IDS — attach the real cluster number to each clean cluster
# ============================================================
for c in clean_clusters:
    match = clusters.index[clusters['min'] == c['w_start'] + PAD]
    c['cluster_id'] = int(match[0])

print([c['cluster_id'] for c in clean_clusters])

[0, 14, 21, 28]


In [ ]:
# ============================================================
# BLOCK R — the rule as a pure function
# ============================================================
def rule_fires(b_fanout, w_fanout):
    """True if a computer with this baseline and window fanout should be flagged."""
    ratio = (w_fanout + 1) / (b_fanout + 1)
    increase = w_fanout-b_fanout
    return w_fanout >= MIN_DESTINATIONS and (ratio >= RATIO_THRESHOLD or increase >= MIN_ABSOLUTE_INCREASE)


In [ ]:
# ============================================================
# BLOCK S1 — alert time per computer (streaming version of the same rule)
# ============================================================
def get_alert_times(cluster_i):
    """Returns {computer: time of first alert} for one cluster."""
    running = collections.defaultdict(set)
    alert_time = {}
    for (t, su, sc, dc) in window_rows[cluster_i]:      # rows are already in time order
        running[sc].add(dc)
        if sc in alert_time:
            continue                                     # already alerted: keep the first time
        b = len(baseline_sets[cluster_i].get(sc, set()))
        if rule_fires(b, len(running[sc])):
            alert_time[sc] = t
    return alert_time

for i, c in enumerate(clean_clusters):
    alerts = get_alert_times(i)
    same = set(alerts) == get_flagged_computers(i)
    print(f"cluster {c['cluster_id']}: {len(alerts)} computers alerted, matches batch: {same}")

cluster 0: 5 computers alerted, matches batch: True
cluster 14: 1252 computers alerted, matches batch: True
cluster 21: 45 computers alerted, matches batch: True
cluster 28: 385 computers alerted, matches batch: True


In [ ]:
# ============================================================
# BLOCK S2 — Full and Static (fixed, seeded 20% of computers)
# ============================================================
import random

SAMPLE_SHARE = 0.20
SEED = 42

# 1. Pick the static sample ONCE, from every computer seen in any window
all_computers = sorted({sc for rows in window_rows for (t, su, sc, dc) in rows})
k = round(SAMPLE_SHARE * len(all_computers))
static_sample = set(random.Random(SEED).sample(all_computers, k))
print(f"{len(all_computers):,} computers seen, {len(static_sample):,} in the static sample")

# 2. Count, per cluster, what Full and Static would store
results = []
for i, c in enumerate(clean_clusters):
    full_rows = static_rows = 0
    attack_rows = attack_kept = 0
    attackers, attackers_covered = set(), set()
    for (t, su, sc, dc) in window_rows[i]:
        in_sample = sc in static_sample
        is_attack = (t, su, sc, dc) in redteam_tuples
        full_rows += 1
        static_rows += in_sample                  # True counts as 1, False as 0
        if is_attack:
            attack_rows += 1
            attack_kept += in_sample
            attackers.add(sc)
            if in_sample:
                attackers_covered.add(sc)
    results.append({
        "cluster": c["cluster_id"],
        "full_rows": full_rows,
        "static_rows": static_rows,
        "static_cost_%": round(100 * static_rows / full_rows, 1),
        "attack_rows": attack_rows,
        "static_attack_kept": attack_kept,
        "attackers": len(attackers),
        "attackers_with_evidence": len(attackers_covered),
    })

s2_table = pd.DataFrame(results)
print(s2_table.to_string(index=False))

12,537 computers seen, 2,507 in the static sample
 cluster  full_rows  static_rows  static_cost_%  attack_rows  static_attack_kept  attackers  attackers_with_evidence
       0     976728       209053           21.4           10                   0          1                        0
      14    4058548       862276           21.2          146                   0          1                        0
      21     193007        40765           21.1            4                   1          2                        1
      28    2257170       489054           21.7            8                   0          1                        0


In [ ]:
# ============================================================
# BLOCK S3 — Adaptive: static 20% + escalation for alerted computers
# ============================================================
ESCALATION_N_MIN = 30     # keep detailed rows for 30 min after an alert
BUFFER_MIN = 5            # pre-alert buffer: also keep the 5 min before the alert

alerts_by_cluster = [get_alert_times(i) for i in range(len(clean_clusters))]

def run_adaptive(n_min, buffer_min):
    N, B = n_min * 60, buffer_min * 60
    out = []
    for i, c in enumerate(clean_clusters):
        alerts = alerts_by_cluster[i]
        stored = attack_rows = attack_kept = 0
        attackers, covered = set(), set()
        for (t, su, sc, dc) in window_rows[i]:
            a = alerts.get(sc)                                   # alert time, or None
            escalated = a is not None and (a - B) <= t <= (a + N)
            keep = (sc in static_sample) or escalated
            stored += keep
            if (t, su, sc, dc) in redteam_tuples:
                attack_rows += 1
                attack_kept += keep
                attackers.add(sc)
                if keep:
                    covered.add(sc)
        out.append({
            "cluster": c["cluster_id"],
            "buffer_min": buffer_min,
            "computers_escalated": len(alerts),
            "adaptive_rows": stored,
            "adaptive_cost_%": round(100 * stored / len(window_rows[i]), 1),
            "attack_rows": attack_rows,
            "attack_kept": attack_kept,
            "attackers": len(attackers),
            "attackers_with_evidence": len(covered),
        })
    return pd.DataFrame(out)

s3_no_buffer = run_adaptive(ESCALATION_N_MIN, 0)
s3_buffer    = run_adaptive(ESCALATION_N_MIN, BUFFER_MIN)
print("ADAPTIVE, no buffer\n", s3_no_buffer.to_string(index=False))
print("\nADAPTIVE, 5-min buffer\n", s3_buffer.to_string(index=False))

ADAPTIVE, no buffer
  cluster  buffer_min  computers_escalated  adaptive_rows  adaptive_cost_%  attack_rows  attack_kept  attackers  attackers_with_evidence
       0           0                    5         209243             21.4           10            1          1                        1
      14           0                 1252         943210             23.2          146            5          1                        1
      21           0                   45          41787             21.7            4            1          2                        1
      28           0                  385         503838             22.3            8            1          1                        1

ADAPTIVE, 5-min buffer
  cluster  buffer_min  computers_escalated  adaptive_rows  adaptive_cost_%  attack_rows  attack_kept  attackers  attackers_with_evidence
       0           5                    5         209290             21.4           10            1          1                        1
  

In [ ]:
# ============================================================
# BLOCK S3-DIAG — per attacker: attack span vs alert time
# ============================================================
diag = []
for i, c in enumerate(clean_clusters):
    alerts = alerts_by_cluster[i]
    atk = [(t, sc) for (t, su, sc, dc) in window_rows[i] if (t, su, sc, dc) in redteam_tuples]
    for comp in sorted({sc for _, sc in atk}):
        times = [t for t, sc in atk if sc == comp]
        a = alerts.get(comp)
        diag.append({
            "cluster": c["cluster_id"],
            "attacker": comp,
            "attack_rows": len(times),
            "attack_span_min": round((max(times) - min(times)) / 60, 1),
            "alerted": a is not None,
            "alert_delay_min": None if a is None else round((a - min(times)) / 60, 1),
            "rows_before_alert": None if a is None else sum(t < a for t in times),
            "rows_after_alert_plus_N": None if a is None else sum(t > a + ESCALATION_N_MIN * 60 for t in times),
        })
s3_diag = pd.DataFrame(diag)
print(s3_diag.to_string(index=False))

 cluster attacker  attack_rows  attack_span_min  alerted  alert_delay_min  rows_before_alert  rows_after_alert_plus_N
       0   C17693           10             96.2     True             86.2                9.0                      0.0
      14   C17693          146            264.2     True              5.7                2.0                    139.0
      21   C17693            3              2.4    False              NaN                NaN                      NaN
      21   C19932            1              0.0    False              NaN                NaN                      NaN
      28   C17693            8            156.0     True             99.7                4.0                      3.0


In [ ]:
# ============================================================
# BLOCK S4 — sensitivity: escalation length N and buffer
# ============================================================
full_total = sum(len(r) for r in window_rows)

grid = []
for n_min in [30, 120, 480, 100000]:          # 100000 min ≈ "stay escalated to the end"
    for buffer_min in [0, 30, 90]:
        df = run_adaptive(n_min, buffer_min)
        grid.append({
            "N_min": "until end" if n_min == 100000 else n_min,
            "buffer_min": buffer_min,
            "cost_%": round(100 * df["adaptive_rows"].sum() / full_total, 1),
            "attack_kept": int(df["attack_kept"].sum()),
            "kept_%": round(100 * df["attack_kept"].sum() / df["attack_rows"].sum(), 1),
        })
        print(f"done N={n_min}, buffer={buffer_min}")

s4_grid = pd.DataFrame(grid)
print(s4_grid.to_string(index=False))

done N=30, buffer=0
done N=30, buffer=30
done N=30, buffer=90
done N=120, buffer=0
done N=120, buffer=30
done N=120, buffer=90
done N=480, buffer=0
done N=480, buffer=30
done N=480, buffer=90
done N=100000, buffer=0
done N=100000, buffer=30
done N=100000, buffer=90
    N_min  buffer_min  cost_%  attack_kept  kept_%
       30           0    22.7            8     4.8
       30          30    23.2           15     8.9
       30          90    23.8           22    13.1
      120           0    23.8           55    32.7
      120          30    24.3           62    36.9
      120          90    24.9           69    41.1
      480           0    24.5          150    89.3
      480          30    25.0          157    93.5
      480          90    25.6          164    97.6
until end           0    24.5          150    89.3
until end          30    25.0          157    93.5
until end          90    25.6          164    97.6


In [ ]:
# ============================================================
# BLOCK SAVE-RESULTS
# ============================================================
OUT = "/content/drive/MyDrive/cyberflight"
s2_table.to_csv(f"{OUT}/s2_static.csv", index=False)
s3_no_buffer.to_csv(f"{OUT}/s3_adaptive_no_buffer.csv", index=False)
s3_buffer.to_csv(f"{OUT}/s3_adaptive_buffer5.csv", index=False)
s3_diag.to_csv(f"{OUT}/s3_diag.csv", index=False)
s4_grid.to_csv(f"{OUT}/s4_grid.csv", index=False)
print("saved")

saved


In [ ]:
# ============================================================
# BLOCK S5 — final policy comparison (per cluster + pooled)
# ============================================================
full_by_cluster = dict(zip(s2_table["cluster"], s2_table["full_rows"]))

runs = {
    "Adaptive v1 (N=30m, no buffer)": run_adaptive(30, 0),
    "Adaptive (N=8h, no buffer)":     run_adaptive(480, 0),
    "Adaptive (N=8h, 90m buffer)":    run_adaptive(480, 90),
}

rows = []
for _, r in s2_table.iterrows():
    rows.append({"cluster": r["cluster"], "policy": "Full",
                 "stored": r["full_rows"], "kept": r["attack_rows"], "attack_rows": r["attack_rows"]})
    rows.append({"cluster": r["cluster"], "policy": "Static 20%",
                 "stored": r["static_rows"], "kept": r["static_attack_kept"], "attack_rows": r["attack_rows"]})
for name, df in runs.items():
    for _, r in df.iterrows():
        rows.append({"cluster": r["cluster"], "policy": name,
                     "stored": r["adaptive_rows"], "kept": r["attack_kept"], "attack_rows": r["attack_rows"]})

final = pd.DataFrame(rows)
final["full"] = final["cluster"].map(full_by_cluster)

# Pooled across all 4 clusters
pooled = final.groupby("policy", sort=False)[["stored", "full", "kept", "attack_rows"]].sum()
pooled["cost_%"] = (100 * pooled["stored"] / pooled["full"]).round(1)
pooled["kept_%"] = (100 * pooled["kept"] / pooled["attack_rows"]).round(1)
print("POOLED (all 4 clusters)\n", pooled[["cost_%", "kept", "attack_rows", "kept_%"]].to_string())

# Per cluster: % of attack rows kept
final["kept_%"] = (100 * final["kept"] / final["attack_rows"]).round(1)
per_cluster = final.pivot(index="policy", columns="cluster", values="kept_%")
per_cluster.columns = [f"c{c}" + (" (tuning)" if c == 0 else "") for c in per_cluster.columns]
print("\nKEPT % PER CLUSTER\n", per_cluster.to_string())

POOLED (all 4 clusters)
                                 cost_%   kept  attack_rows  kept_%
policy                                                            
Full                             100.0  168.0        168.0   100.0
Static 20%                        21.4    1.0        168.0     0.6
Adaptive v1 (N=30m, no buffer)    22.7    8.0        168.0     4.8
Adaptive (N=8h, no buffer)        24.5  150.0        168.0    89.3
Adaptive (N=8h, 90m buffer)       25.6  164.0        168.0    97.6

KEPT % PER CLUSTER
                                 c0.0 (tuning)  c14.0  c21.0  c28.0
policy                                                            
Adaptive (N=8h, 90m buffer)             100.0  100.0   25.0   87.5
Adaptive (N=8h, no buffer)               10.0   98.6   25.0   50.0
Adaptive v1 (N=30m, no buffer)           10.0    3.4   25.0   12.5
Full                                    100.0  100.0  100.0  100.0
Static 20%                                0.0    0.0   25.0    0.0


In [ ]:
# ============================================================
# BLOCK CL — computer-level grading per cluster
# ============================================================
cl_rows = []
for i, c in enumerate(clean_clusters):
    alerted = set(alerts_by_cluster[i])
    attackers = {sc for (t, su, sc, dc) in window_rows[i] if (t, su, sc, dc) in redteam_tuples}
    caught = alerted & attackers
    cl_rows.append({
        "cluster": c["cluster_id"],
        "role": "tuning" if c["cluster_id"] == 0 else "test",
        "computers_alerted": len(alerted),
        "attackers": len(attackers),
        "attackers_caught": len(caught),
        "false_alarm_computers": len(alerted - attackers),
        "computer_precision_%": round(100 * len(caught) / len(alerted), 2) if alerted else 0.0,
    })
cl_table = pd.DataFrame(cl_rows)
print(cl_table.to_string(index=False))

 cluster   role  computers_alerted  attackers  attackers_caught  false_alarm_computers  computer_precision_%
       0 tuning                  5          1                 1                      4                 20.00
      14   test               1252          1                 1                   1251                  0.08
      21   test                 45          2                 0                     45                  0.00
      28   test                385          1                 1                    384                  0.26


Here's each block, plain-language first, then what the code is literally doing.

Block 1 — Setup

What it's for: connects Colab to your Google Drive (where the data files live) and imports the libraries you'll use everywhere else.

drive.mount(...) — lets this notebook read files sitting in your Drive.
import gzip — the data files are compressed (.gz), this lets Python open them without manually unzipping first.
import pandas as pd — the table/dataframe library you use throughout.
The two file paths are just saved as variables so you don't retype them in every block.
Block 2 — Load Red Team ground truth

What it's for: loads the "answer key" — the list of logins that were actually the attacker — into a usable form.

Opens redteam.txt.gz and reads every line into a plain list of text.
Turns that list into a pandas table (redteam_df) with four named columns, since the raw file has no header row telling you what each column means.
redteam_tuples repackages each row as a tuple (time, user, source, destination) and puts all of them into a Python set. A set lets you check "is this exact combination an attack?" instantly, instead of searching through the table row by row every time — this is what makes the later streaming code fast.
Block 3 — Build attack clusters, pick a spread-out test set

What it's for: groups the 749 individual attack events into clusters (bursts that happened close together in time), then picks 6 of those clusters spread across the whole 58-day span — so you're testing on early, middle, and late attacks, not just the first one.

gap = ...diff() — calculates the time gap between each attack and the one before it.
(gap > 3600).cumsum() — starts a new cluster number every time there's more than a 1-hour gap. This is how "35 separate attack clusters" gets identified from 749 individual rows.
clusters = ...agg(['min','max','count']) — for each cluster, records when it starts, when it ends, and how many attack events are in it.
chosen_idx — picks cluster index 0 (the very start), then roughly 20%, 40%, 60%, 80%, and the very last one — a deliberate spread across the timeline, not a random or convenient pick.
Block 4 — Trigger rule: Fanout

What it's for: this is the actual detector. For each of your 6 chosen clusters, it figures out which computers suddenly started talking to far more destinations than they normally do.

Step by step:

For each chosen cluster, it defines two time ranges: the window (the attack period itself, padded a bit on either side) and the baseline (a 45,000-second stretch right before the window — "what normal looks like" for comparison).
contaminated checks whether that baseline accidentally contains a different real attack — if it does, that cluster gets skipped, because you can't call something "normal behavior" if it secretly contains another attack.
It streams through the entire auth.txt.gz file line by line (never loading the whole 1-billion-row file into memory at once), stopping once it passes the last cluster's end time — no point reading further.
For every login where logon_type == 'Network' (computer-to-computer connections, not interactive logins), it checks: does this row's time fall inside a baseline range or a window range for any of the 6 clusters? If so, it records which destination computers that source computer talked to, separately for baseline vs. window.
get_flagged_computers() is the actual rule: for a given cluster, compare each computer's window activity to its baseline. If a computer suddenly reaches 5 or more destinations, and either that's 5x its normal amount or 6 more destinations than usual — flag it as suspicious.

This function is the one piece everything downstream depends on — it's your "is this computer behaving suspiciously right now?" check.

Block 5 — Grade the trigger

What it's for: a sanity check — confirms the rule still performs as expected (97.6% recall) before you build anything on top of it.

For each cluster, it calls get_flagged_computers() to get the suspicious list, then goes through every login that happened in that cluster's window.
For each login, it checks two things: was this computer flagged? And was this login actually a real attack? Based on those two yes/no answers, it sorts the event into one of four buckets — correctly caught (TP), false alarm (FP), missed attack (FN), or correctly ignored (TN).
At the end, it converts those counts into precision ("of everything I flagged, how much was real?") and recall ("of all real attacks, how much did I catch?").
Block 6 — Known limitation

What it's for: not code that runs anything new — just a clearly labeled note in the notebook, documenting the one thing you know this rule doesn't catch (the cluster 21 case, where an attacker reused the same single connection repeatedly instead of branching out to new ones), so it's visible to anyone reading the notebook rather than silently forgotten.

In [3]:
import os, glob, pickle, gzip
import pandas as pd

# ---- 1. Make sure Drive is properly mounted ----
ROOT = "/content/drive/MyDrive"
if not os.path.exists(ROOT) or len(os.listdir(ROOT)) == 0:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
print("Drive items visible:", len(os.listdir(ROOT)))

# ---- 2. Find the input files wherever they are ----
def find_file(name):
    hits = glob.glob(f"{ROOT}/**/{name}", recursive=True)
    return hits[0] if hits else None

REDTEAM_PATH = find_file("redteam.txt.gz")
STATE_FILE   = find_file("stream_state.pkl")
print("redteam.txt.gz   ->", REDTEAM_PATH)
print("stream_state.pkl ->", STATE_FILE)
assert REDTEAM_PATH, "redteam.txt.gz not found in Drive (see note below)."
assert STATE_FILE, "stream_state.pkl not found: run the Block 4 stream + save cell first."

OUT = f"{ROOT}/cyberflight/export"
os.makedirs(OUT, exist_ok=True)

# ---- 3. Red team answer key ----
cols = ['time', 'user', 'source_computer', 'destination_computer']
with gzip.open(REDTEAM_PATH, 'rt', encoding='utf-8') as f:
    redteam_df = pd.read_csv(f, header=None, names=cols)
redteam_df['time'] = pd.to_numeric(redteam_df['time'])
redteam_tuples = set(zip(redteam_df['time'].astype(int), redteam_df['user'],
                         redteam_df['source_computer'], redteam_df['destination_computer']))
print("red team events:", len(redteam_df))   # expect 749

# ---- 4. Rebuild the 35 clusters ----
rt_sorted = redteam_df.sort_values('time').reset_index(drop=True)
rt_sorted['cluster'] = (rt_sorted['time'].diff().fillna(0) > 3600).cumsum()
clusters = rt_sorted.groupby('cluster')['time'].agg(['min', 'max', 'count'])

# ---- 5. Reload the saved streaming state ----
with open(STATE_FILE, 'rb') as f:
    state = pickle.load(f)
clean_clusters = state['clean_clusters']
baseline_sets  = state['baseline_sets']
window_rows    = state['window_rows']
PAD = state['settings']['PAD']
for c in clean_clusters:
    c['cluster_id'] = int(clusters.index[clusters['min'] == c['w_start'] + PAD][0])
print("clusters:", [c['cluster_id'] for c in clean_clusters])   # expect [0, 14, 21, 28]

# ---- 6. Export the four files ----
frames = []
for i, c in enumerate(clean_clusters):
    df = pd.DataFrame(window_rows[i], columns=["time", "user", "src", "dst"])
    df["cluster"] = c["cluster_id"]
    df["is_attack"] = [(t, u, s, d) in redteam_tuples
                       for t, u, s, d in zip(df["time"], df["user"], df["src"], df["dst"])]
    frames.append(df)
rows = pd.concat(frames, ignore_index=True)
for col in ["user", "src", "dst"]:
    rows[col] = rows[col].astype("category")
rows.to_parquet(f"{OUT}/window_rows.parquet", index=False)

pd.DataFrame([{"cluster": c["cluster_id"], "src": comp, "baseline_fanout": len(dsts)}
              for i, c in enumerate(clean_clusters)
              for comp, dsts in baseline_sets[i].items()]
             ).to_parquet(f"{OUT}/baseline_fanout.parquet", index=False)

pd.DataFrame([{"cluster": c["cluster_id"], "w_start": c["w_start"], "w_end": c["w_end"],
               "b_start": c["b_start"], "b_end": c["b_end"]} for c in clean_clusters]
             ).to_csv(f"{OUT}/clusters.csv", index=False)
redteam_df.to_csv(f"{OUT}/redteam.csv", index=False)

print(rows.shape, int(rows["is_attack"].sum()), "attack rows")   # expect (7485453, 6) and 168
for fn in os.listdir(OUT):
    print(fn, round(os.path.getsize(f"{OUT}/{fn}") / 1e6, 1), "MB")

Drive items visible: 8
redteam.txt.gz   -> /content/drive/MyDrive/redteam.txt.gz
stream_state.pkl -> /content/drive/MyDrive/cyberflight/stream_state.pkl
red team events: 749
clusters: [0, 14, 21, 28]
(7485453, 6) 168 attack rows
window_rows.parquet 39.7 MB
baseline_fanout.parquet 0.2 MB
clusters.csv 0.0 MB
redteam.csv 0.0 MB
